# Foundation Backbone Smoke Test

Exercises all backbones exposed through `ultralytics.nn.modules.foundation_backbones`:

- DINOv2-S/14 (no registers)
- DINOv2-B/14 (no registers)
- DINOv3-S/16 (license-gated by Meta)
- AM-RADIO v2.5-B

For each backbone this notebook checks, in order:

1. **Adapter-level forward** — shape, dtype, finiteness of the raw `[B, C, Hf, Wf]` feature map.
2. **Freeze behavior** — `requires_grad=False` on all backbone params, and `.train()` cannot pull the backbone out of `eval()`.
3. **Full detector forward** — the backbone's model YAML built via `ultralytics.YOLO(...)` and run end-to-end through the `Detect` head.

Any backbone whose pretrained weights are unavailable (no network, or gated access — DINOv3 requires accepting Meta's license) reports **SKIPPED** with the original error instead of crashing the notebook or silently falling back to random weights. Run cells top to bottom; the summary table at the end reflects whatever actually ran in this environment.

In [1]:
import time
import traceback

import torch

from ultralytics import YOLO
from ultralytics.nn.modules.foundation_backbones import (
    FoundationBackboneError,
    count_parameters,
)
from ultralytics.nn.modules.foundation_backbones.dinov2 import DINOv2Backbone
from ultralytics.nn.modules.foundation_backbones.dinov3 import DINOv3Backbone
from ultralytics.nn.modules.foundation_backbones.radio import RADIOBackbone

torch.manual_seed(0)

# imgsz=448 is divisible by 14 (DINOv2 patch), 16 (DINOv3/RADIO patch), and 32
# (Ultralytics' own stride bookkeeping) — the low-memory smoke-test size recommended
# in the README.
IMGSZ = 448

results = []  # list of dicts, one row per backbone, printed as a summary table at the end

In [2]:
def check_adapter(name, backbone_cls, **kwargs):
    """Instantiate one FoundationBackbone adapter directly and validate its forward pass
    and freeze bookkeeping. Returns a result dict; never raises."""
    row = {"backbone": name, "stage": "adapter", "status": "FAIL", "detail": ""}
    t0 = time.time()
    try:
        backbone = backbone_cls(freeze=True, **kwargs)
    except FoundationBackboneError as e:
        row["status"] = "SKIPPED"
        row["detail"] = f"weights unavailable: {e}"
        return row, None

    x = torch.randn(1, 3, IMGSZ, IMGSZ)
    with torch.no_grad():
        out = backbone(x)

    assert out.ndim == 4, f"expected 4D output, got {out.shape}"
    assert out.shape[0] == 1
    assert out.shape[1] == backbone.out_channels, (out.shape[1], backbone.out_channels)
    expected_spatial = IMGSZ // backbone.patch_stride
    assert out.shape[2] == expected_spatial and out.shape[3] == expected_spatial, out.shape
    assert torch.isfinite(out).all(), "non-finite values in feature map"
    assert all(not p.requires_grad for p in backbone.parameters()), "frozen backbone has trainable params"

    backbone.train(True)  # must not pull a frozen backbone back into train mode
    assert backbone.training is False, "frozen backbone entered train() mode"

    row["status"] = "PASS"
    row["detail"] = (
        f"out_channels={backbone.out_channels} patch_stride={backbone.patch_stride} "
        f"feature_map={tuple(out.shape)} time={time.time() - t0:.2f}s"
    )
    return row, backbone

In [3]:
def check_full_detector(name, config_path):
    """Build the full YOLO detector from a model YAML and run one forward pass through
    the Detect head. Returns a result dict; never raises."""
    row = {"backbone": name, "stage": "full_detector", "status": "FAIL", "detail": ""}
    t0 = time.time()
    try:
        model = YOLO(config_path)
    except FoundationBackboneError as e:
        row["status"] = "SKIPPED"
        row["detail"] = f"weights unavailable: {e}"
        return row

    x = torch.randn(1, 3, IMGSZ, IMGSZ)
    with torch.no_grad():
        preds = model.model(x)
    assert preds is not None

    report = count_parameters(model.model)
    row["status"] = "PASS"
    row["detail"] = (
        f"stride={model.model.stride.tolist()} "
        f"backbone_params={report['backbone_parameters']:,} "
        f"trainable_backbone_params={report['trainable_backbone_parameters']:,} "
        f"time={time.time() - t0:.2f}s"
    )
    return row

## DINOv2-S/14 (no registers)

In [4]:
row, _ = check_adapter("DINOv2-S/14", DINOv2Backbone, variant="vits14", pretrained=True)
results.append(row)
print(row)

Using cache found in /home/artur/.cache/torch/hub/facebookresearch_dinov2_main
/home/artur/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/artur/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/artur/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


{'backbone': 'DINOv2-S/14', 'stage': 'adapter', 'status': 'PASS', 'detail': 'out_channels=384 patch_stride=14 feature_map=(1, 384, 32, 32) time=0.50s'}


In [5]:
row = check_full_detector("DINOv2-S/14", "foundation_backbone_configs/yolo_dinov2_vits14.yaml")
results.append(row)
print(row)

WARNING ⚠️ no model scale passed. Assuming scale='n'.


Using cache found in /home/artur/.cache/torch/hub/facebookresearch_dinov2_main


{'backbone': 'DINOv2-S/14', 'stage': 'full_detector', 'status': 'PASS', 'detail': 'stride=[14.0] backbone_params=22,056,576 trainable_backbone_params=0 time=0.52s'}


## DINOv2-B/14 (no registers)

In [6]:
row, _ = check_adapter("DINOv2-B/14", DINOv2Backbone, variant="vitb14", pretrained=True)
results.append(row)
print(row)

Using cache found in /home/artur/.cache/torch/hub/facebookresearch_dinov2_main


{'backbone': 'DINOv2-B/14', 'stage': 'adapter', 'status': 'PASS', 'detail': 'out_channels=768 patch_stride=14 feature_map=(1, 768, 32, 32) time=1.28s'}


In [7]:
row = check_full_detector("DINOv2-B/14", "foundation_backbone_configs/yolo_dinov2_vitb14.yaml")
results.append(row)
print(row)

WARNING ⚠️ no model scale passed. Assuming scale='n'.


Using cache found in /home/artur/.cache/torch/hub/facebookresearch_dinov2_main


{'backbone': 'DINOv2-B/14', 'stage': 'full_detector', 'status': 'PASS', 'detail': 'stride=[14.0] backbone_params=86,580,480 trainable_backbone_params=0 time=1.63s'}


## DINOv3-S/16

DINOv3 weights are access-gated by Meta. Accept the license at
https://github.com/facebookresearch/dinov3 and ensure hub download access (or pass a local
`weights=` path to `DINOv3Backbone`) for this to load. Otherwise it reports SKIPPED below
— it will never silently fall back to random weights.

In [8]:
row, _ = check_adapter("DINOv3-S/16", DINOv3Backbone, variant="vits16", pretrained=True)
results.append(row)
print(row)

Using cache found in /home/artur/.cache/torch/hub/facebookresearch_dinov3_main


Downloading: "https://dl.fbaipublicfiles.com/dinov3/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth" to /home/artur/.cache/torch/hub/checkpoints/dinov3_vits16_pretrain_lvd1689m-08c60483.pth
{'backbone': 'DINOv3-S/16', 'stage': 'adapter', 'status': 'SKIPPED', 'detail': "weights unavailable: Failed to load pretrained DINOv3 weights for 'dinov3_vits16' via torch.hub.load('facebookresearch/dinov3', ...). DINOv3 weights are access-gated by Meta — ensure you've accepted the license and either have hub download access or pass an explicit local weights= path. Refusing to silently fall back to a randomly initialized model. Original error: <HTTPError 403: 'Forbidden'>"}


In [9]:
row = check_full_detector("DINOv3-S/16", "foundation_backbone_configs/yolo_dinov3_vits16.yaml")
results.append(row)
print(row)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
Downloading: "https://dl.fbaipublicfiles.com/dinov3/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth" to /home/artur/.cache/torch/hub/checkpoints/dinov3_vits16_pretrain_lvd1689m-08c60483.pth


Using cache found in /home/artur/.cache/torch/hub/facebookresearch_dinov3_main


{'backbone': 'DINOv3-S/16', 'stage': 'full_detector', 'status': 'SKIPPED', 'detail': "weights unavailable: Failed to load pretrained DINOv3 weights for 'dinov3_vits16' via torch.hub.load('facebookresearch/dinov3', ...). DINOv3 weights are access-gated by Meta — ensure you've accepted the license and either have hub download access or pass an explicit local weights= path. Refusing to silently fall back to a randomly initialized model. Original error: <HTTPError 403: 'Forbidden'>"}


## AM-RADIO v2.5-B

In [10]:
row, _ = check_adapter("RADIO-v2.5-B", RADIOBackbone, variant="radio_v2.5-b", pretrained=True)
results.append(row)
print(row)

Using cache found in /home/artur/.cache/torch/hub/NVlabs_RADIO_main


{'backbone': 'RADIO-v2.5-B', 'stage': 'adapter', 'status': 'PASS', 'detail': 'out_channels=768 patch_stride=16 feature_map=(1, 768, 28, 28) time=3.71s'}


In [11]:
row = check_full_detector("RADIO-v2.5-B", "foundation_backbone_configs/yolo_radio_v25_b.yaml")
results.append(row)
print(row)

WARNING ⚠️ no model scale passed. Assuming scale='n'.


Using cache found in /home/artur/.cache/torch/hub/NVlabs_RADIO_main


{'backbone': 'RADIO-v2.5-B', 'stage': 'full_detector', 'status': 'PASS', 'detail': 'stride=[16.0] backbone_params=98,233,344 trainable_backbone_params=0 time=1.02s'}


## Summary

In [12]:
try:
    import pandas as pd

    df = pd.DataFrame(results)[["backbone", "stage", "status", "detail"]]
    display(df)
except ImportError:
    for row in results:
        print(f"{row['backbone']:16s} {row['stage']:14s} {row['status']:8s} {row['detail']}")

n_fail = sum(r["status"] == "FAIL" for r in results)
n_pass = sum(r["status"] == "PASS" for r in results)
n_skip = sum(r["status"] == "SKIPPED" for r in results)
print(f"\n{n_pass} passed, {n_skip} skipped, {n_fail} failed")
assert n_fail == 0, "one or more smoke tests failed — see the table above"

,backbone,stage,status,detail
0,DINOv2-S/14,adapter,PASS,out_channels=384 patch_stride=14 feature_map=(...
1,DINOv2-S/14,full_detector,PASS,"stride=[14.0] backbone_params=22,056,576 train..."
2,DINOv2-B/14,adapter,PASS,out_channels=768 patch_stride=14 feature_map=(...
3,DINOv2-B/14,full_detector,PASS,"stride=[14.0] backbone_params=86,580,480 train..."
4,DINOv3-S/16,adapter,SKIPPED,weights unavailable: Failed to load pretrained...
5,DINOv3-S/16,full_detector,SKIPPED,weights unavailable: Failed to load pretrained...
6,RADIO-v2.5-B,adapter,PASS,out_channels=768 patch_stride=16 feature_map=(...
7,RADIO-v2.5-B,full_detector,PASS,"stride=[16.0] backbone_params=98,233,344 train..."



6 passed, 2 skipped, 0 failed
